# 06. Quasi-equilibrium levels and their error

A quasi-equilibrium (QE) treatment picks slow steps and equilibrates the rest.
`qe_error.py` checks that choice against the steady-state approximation (SSA):
it computes the QE levels, their error $\log_{10}(j_\mathrm{level}/j_\mathrm{SSA})$,
and where that error comes from.

- **QE″**: one slow step per independent route, all other steps equilibrated,
  every intermediate in the site balance, slow-step reverse rates kept. Works for
  any mechanism.
- **QE′** (serial cycles): states up to the slow step $s$ equilibrated with the
  empty site, downstream states set to zero, reverse rate of $s$ dropped.
- **QE** (serial cycles): the reactant of $s$ saturated, so $r = k_s'$.

All errors below are in decades.

In [1]:
import sys
sys.path.insert(0, "..")          # qe_error.py lives in the repository root

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import qe_error as E
import mechanisms as M

## (a) Serial OER with ET4 as the rate-limiting step

For a serial cycle with slow step $s$, let $w_j=\prod_{i<j}K_i$, $Z=\prod_i K_i$,
$A_s=\sum_{j\le s}w_j$ and $B_s=Z^{-1}\sum_{j>s}w_j$. Then
$r_{\mathrm{QE''},s}=k_s w_s(1-1/Z)/(A_s+B_s)$ and $X_s=r/r_{\mathrm{QE''},s}$.
The errors split exactly into four terms:

| term | expression | in |
|---|---|---|
| rate control | $-\log X_s$ | QE″, QE′, QE |
| reverse rate | $-\log(1-1/Z)$ | QE′, QE |
| downstream | $-\log(1-\Theta_\mathrm{downstream})$ | QE′, QE |
| fixed coverage | $-\log\theta_s^{\mathrm{QE'}}$ | QE |

In [2]:
mech = M.oer_case("ET4")
etas = np.linspace(0.05, 0.50, 10)
net, rows = E.scan(mech, etas, ["ET4"])
E.print_report(net, ["ET4"], rows)

# routes: 1  [+1 +1 +1 +1 +1]  (coefficients over ET1, ET2, ET3, ChemPT, ET4)
# QE'' slow: {ET4}   equilibrated: {ET1, ET2, ET3, ChemPT}
# degree of rate control X_i (SSA)
  eta/V       j_SSA        ET1      ET2      ET3   ChemPT      ET4
  0.050   3.258e-12     0.0000   0.0000   0.0000   0.0000   1.0000
  0.100   1.720e-09     0.0000   0.0000   0.0000   0.0001   0.9999
  0.150   5.066e-08     0.0000   0.0000   0.0000   0.0002   0.9998
  0.200   1.795e-07     0.0000   0.0000   0.0000   0.0005   0.9995
  0.250   4.920e-07     0.0000   0.0000   0.0000   0.0013   0.9987
  0.300   1.306e-06     0.0000   0.0000   0.0000   0.0034   0.9966
  0.350   3.438e-06     0.0000   0.0000   0.0000   0.0089   0.9911
  0.400   8.967e-06     0.0000   0.0000   0.0000   0.0232   0.9767
  0.450   2.285e-05     0.0000   0.0000   0.0000   0.0592   0.9408
  0.500   5.510e-05     0.0000   0.0000   0.0000   0.1428   0.8572
# currents (A/cm^2) and error vs SSA, log10(j_level/j_SSA) in decades
  eta/V       j_SSA  

In [3]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
eta = [r["eta"] for r in rows]
for key, lab in (("err_qepp", "QE″"), ("err_qep", "QE′"), ("err_qe", "QE")):
    a1.plot(eta, [r[key] for r in rows], "o-", label=lab)
a1.set_xlabel(r"$\eta$ / V"); a1.set_ylabel(r"log$_{10}(j/j_\mathrm{SSA})$"); a1.legend()
for key, lab in (("rate_control", "rate control"), ("reverse", "reverse rate"),
                 ("downstream", "downstream"), ("coverage", "fixed coverage")):
    a2.plot(eta, [r["serial"][key] for r in rows], "o-", label=lab)
a2.set_yscale("symlog", linthresh=1e-4)
a2.set_xlabel(r"$\eta$ / V"); a2.set_ylabel("term / decades"); a2.legend()
fig.tight_layout()

Classical QE is two to five decades too high: it saturates MOO⁻, whose QE′
coverage is $10^{-2}$ or less. QE″ and QE′ track the SSA until ET4 loses rate
control at high overpotential, where the rate-control term grows. ET4 is the
last step, so the downstream term is zero.

## (b) Branched 2e⁻/4e⁻ ORR

Four states and five steps give two independent routes, so QE″ needs two slow
steps. Free energies follow scaling relations with $\Delta G_\mathrm{OH}=1.66$ eV
and every intrinsic rate constant is 1 (`M.orr_branched_mechanism`).

For a branched network QE′ is not unique and classical QE is not defined, so
only QE″ is reported. Its error is the integrated rate control of the
equilibrated steps $F$:
$$\ln\frac{J_\mathrm{QE''}}{J}=\int_0^\infty\sum_{j\in F}X_j(\lambda)\,d\ln\lambda,$$
where $k_j$ and $k_{-j}$ of every $j\in F$ are scaled by $\lambda$ at fixed $K_j$.
The total is exact. The split among the steps of $F$ depends on the path; here
all of $F$ is scaled together.

In [4]:
orr = M.orr_branched_mechanism()
etas = np.linspace(-0.43, -0.93, 6)
sets = [["PCET2", "PCET3"], ["PCET1", "PCET2"], ["PCET1", "PCET3"]]
results = {}
for slow in sets:
    net, rows = E.scan(orr, etas, slow)
    results[tuple(slow)] = rows
    print()
    E.print_report(net, slow, rows)


# routes: 2  [+1 +1 +0 +0 +0]  [+1 +0 +1 +1 +1]  (coefficients over PCET1, PCET2, PCET3, PCET4, PCET5)
# QE'' slow: {PCET2, PCET3}   equilibrated: {PCET1, PCET4, PCET5}
# branched network: QE' is not unique and classical QE is not defined; both skipped
# degree of rate control X_i (SSA)
  eta/V       j_SSA      PCET1    PCET2    PCET3    PCET4    PCET5
 -0.430  -1.375e-11     0.0392   0.3132   0.6476   0.0000   0.0000
 -0.530  -1.639e-09     0.6667   0.0000   0.3333   0.0000   0.0000
 -0.630  -1.703e-08     0.9899  -0.1616   0.1717   0.0000   0.0000
 -0.730  -1.205e-07     0.9998  -0.1666   0.1668   0.0000   0.0000
 -0.830  -8.435e-07     1.0000  -0.1667   0.1667   0.0000   0.0000
 -0.930  -5.905e-06     1.0000  -0.1667   0.1667   0.0000   0.0000
# currents (A/cm^2) and error vs SSA, log10(j_level/j_SSA) in decades
  eta/V       j_SSA       j_QE″      err
 -0.430  -1.375e-11  -1.431e-11   0.0174
 -0.530  -1.639e-09  -4.916e-09   0.4771
 -0.630  -1.703e-08  -1.687e-06   1.9958
 -0.730 


# routes: 2  [+1 +1 +0 +0 +0]  [+1 +0 +1 +1 +1]  (coefficients over PCET1, PCET2, PCET3, PCET4, PCET5)
# QE'' slow: {PCET1, PCET2}   equilibrated: {PCET3, PCET4, PCET5}
# branched network: QE' is not unique and classical QE is not defined; both skipped
# degree of rate control X_i (SSA)
  eta/V       j_SSA      PCET1    PCET2    PCET3    PCET4    PCET5
 -0.430  -1.375e-11     0.0392   0.3132   0.6476   0.0000   0.0000
 -0.530  -1.639e-09     0.6667   0.0000   0.3333   0.0000   0.0000
 -0.630  -1.703e-08     0.9899  -0.1616   0.1717   0.0000   0.0000
 -0.730  -1.205e-07     0.9998  -0.1666   0.1668   0.0000   0.0000
 -0.830  -8.435e-07     1.0000  -0.1667   0.1667   0.0000   0.0000
 -0.930  -5.905e-06     1.0000  -0.1667   0.1667   0.0000   0.0000
# currents (A/cm^2) and error vs SSA, log10(j_level/j_SSA) in decades
  eta/V       j_SSA       j_QE″      err
 -0.430  -1.375e-11  -4.681e-10   1.5320
 -0.530  -1.639e-09  -3.277e-09   0.3010
 -0.630  -1.703e-08  -2.294e-08   0.1293
 -0.730 


# routes: 2  [+1 +1 +0 +0 +0]  [+1 +0 +1 +1 +1]  (coefficients over PCET1, PCET2, PCET3, PCET4, PCET5)
# QE'' slow: {PCET1, PCET3}   equilibrated: {PCET2, PCET4, PCET5}
# branched network: QE' is not unique and classical QE is not defined; both skipped
# degree of rate control X_i (SSA)
  eta/V       j_SSA      PCET1    PCET2    PCET3    PCET4    PCET5
 -0.430  -1.375e-11     0.0392   0.3132   0.6476   0.0000   0.0000
 -0.530  -1.639e-09     0.6667   0.0000   0.3333   0.0000   0.0000
 -0.630  -1.703e-08     0.9899  -0.1616   0.1717   0.0000   0.0000
 -0.730  -1.205e-07     0.9998  -0.1666   0.1668   0.0000   0.0000
 -0.830  -8.435e-07     1.0000  -0.1667   0.1667   0.0000   0.0000
 -0.930  -5.905e-06     1.0000  -0.1667   0.1667   0.0000   0.0000
# currents (A/cm^2) and error vs SSA, log10(j_level/j_SSA) in decades
  eta/V       j_SSA       j_QE″      err
 -0.430  -1.375e-11  -2.335e-10   1.2299
 -0.530  -1.639e-09  -1.639e-09   0.0000
 -0.630  -1.703e-08  -1.147e-08  -0.1717
 -0.730 

In [5]:
fig, axs = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, slow in zip(axs, sets):
    rows = results[tuple(slow)]
    eta = [r["eta"] for r in rows]
    ax.plot(eta, [r["err_qepp"] for r in rows], "ko-", label="QE″ error")
    for lab in rows[0]["path"]["shares"]:
        ax.plot(eta, [r["path"]["shares"][lab] for r in rows], "--", label=f"∫X {lab}")
    ax.axhline(0, color="0.6", lw=0.6)
    ax.set_title("QE″ {" + ", ".join(slow) + "}"); ax.set_xlabel(r"$\eta$ / V")
    ax.legend(fontsize=8)
axs[0].set_ylabel("decades")
fig.tight_layout()

In [6]:
for slow in sets[1:]:
    r = results[tuple(slow)][-1]
    print(f"{str(slow):24s} j_QE''/j_SSA at {r['eta']:.2f} V = {10 ** r['err_qepp']:.4f}")

['PCET1', 'PCET2']       j_QE''/j_SSA at -0.93 V = 1.3333
['PCET1', 'PCET3']       j_QE''/j_SSA at -0.93 V = 0.6667


- **{PCET2, PCET3}** keeps both exits slow. Its error is carried by PCET1,
  which loses its equilibrium as the overpotential grows, and reaches about
  seven decades at −0.93 V.
- **{PCET1, PCET2}** and **{PCET1, PCET3}** fail near onset but, once PCET1
  limits, get the current within a factor of 4/3 and 2/3, so they look right
  on a Tafel plot. Each forces all the flux
  through one exit, so the selectivity is wrong. The error sits in the
  equilibrated exit, PCET3 or PCET2.
- Sets that leave PCET1 and PCET2 both equilibrated close a loop and are
  refused:

In [7]:
try:
    E.Network(orr).check_slow(["PCET4", "PCET5"])
except E.QEError as e:
    print("QEError:", e)

QEError: the equilibrated steps {PCET1, PCET2} close a loop: their equilibria would force K_PCET1 * K_PCET2 = 1, which holds at one potential at most. Make one of them slow instead.


## Your own mechanism

Edit the `MECHANISM` block and `SLOW_STEPS` in `qe_error.py` and run
`python qe_error.py`, or call `E.scan(mech, etas, slow)` on any
`core.Mechanism` as above.